In [1]:
from panzer.exchanges.binance.public import BinancePublicClient

## Client

In [2]:
client = BinancePublicClient(market="um", safety_ratio=0.5)

2025-11-23 14:35:12     INFO [panzer.binance.public.um] Inicializando BinancePublicClient(market=um)
2025-11-23 14:35:13     INFO [panzer.binance.public.um] Rate limiter inicializado: max_per_minute=2400 safety_ratio=0.50


In [3]:
client = BinancePublicClient(market="cm", safety_ratio=0.5)

2025-11-23 14:35:13     INFO [panzer.binance.public.cm] Inicializando BinancePublicClient(market=cm)
2025-11-23 14:35:13     INFO [panzer.binance.public.cm] Rate limiter inicializado: max_per_minute=2400 safety_ratio=0.50


In [4]:
client = BinancePublicClient(market="spot", safety_ratio=0.5)

2025-11-23 14:35:13     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2025-11-23 14:35:14     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.50


## Limter test

In [5]:
N = 20
results = []

In [6]:
from time import time
from panzer.rate_limit.binance_fixed import BinanceFixedWindowLimiter
from panzer.http.client import binance_public_get, BINANCE_SPOT_BASE_URL

# relaja las condiciones para agilizar pruebas siguientes
limiter = BinanceFixedWindowLimiter(
    max_per_minute=60,   # límite teórico para esta prueba
    safety_ratio=0.9,    # effective_limit = 54 req/min
)

for i in range(8):
    t0 = time()
    try:
        data, headers = binance_public_get(
            base_url=BINANCE_SPOT_BASE_URL,
            endpoint="/api/v3/time",
            params=None,
            limiter=limiter,
            weight=1,
            timeout=5,
        )
        status = 200
        exc = None
    except Exception as e:
        data, headers = None, None
        status = getattr(e, "status_code", 429)
        exc = repr(e)

    t1 = time()
    elapsed = t1 - t0
    print(
        f"[i={i:02d}] status={status} elapsed={elapsed:.3f}s | "
        f"used_local={limiter.used_local} | "
        f"server_used={limiter.last_server_used} | exc={exc}"
    )

2025-11-23 14:35:14  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=59, weight=1, effective_limit=54). Durmiendo 45.17 segundos.


[i=00] status=200 elapsed=0.254s | used_local=59 | server_used=59 | exc=None
[i=01] status=200 elapsed=45.466s | used_local=1 | server_used=1 | exc=None
[i=02] status=200 elapsed=0.254s | used_local=2 | server_used=2 | exc=None
[i=03] status=200 elapsed=0.254s | used_local=3 | server_used=3 | exc=None
[i=04] status=200 elapsed=0.257s | used_local=4 | server_used=4 | exc=None
[i=05] status=200 elapsed=0.259s | used_local=5 | server_used=5 | exc=None
[i=06] status=200 elapsed=0.259s | used_local=6 | server_used=6 | exc=None
[i=07] status=200 elapsed=0.254s | used_local=7 | server_used=7 | exc=None


## Test agresivo con concurrencia

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from panzer.rate_limit.binance_fixed import BinanceFixedWindowLimiter
from panzer.http.client import binance_public_get, BINANCE_SPOT_BASE_URL

limiter = BinanceFixedWindowLimiter(
    max_per_minute=60,   # límite teórico para esta prueba
    safety_ratio=0.9,    # effective_limit = 54 req/min
)

def worker(idx: int):
    t0 = time()
    data, headers = binance_public_get(
        base_url=BINANCE_SPOT_BASE_URL,
        endpoint="/api/v3/time",
        params=None,
        limiter=limiter,
        weight=1,
        timeout=5,
    )
    print(f"worker {idx} done: data={data}\nheaders={headers}")
    t1 = time()
    elapsed = t1 - t0
    return idx, elapsed, limiter.used_local, limiter.last_server_used

N = 10
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = [pool.submit(worker, i) for i in range(N)]

for fut in as_completed(futures):
    idx, elapsed, used_local, server_used = fut.result()
    print(
        f"[idx={idx:02d}] elapsed={elapsed:.3f}s | "
        f"used_local={used_local} | server_used={server_used}"
    )


worker 0 done: data={'serverTime': 1763904962114}
headers={'Content-Type': 'application/json;charset=UTF-8', 'Content-Length': '28', 'Connection': 'keep-alive', 'Date': 'Sun, 23 Nov 2025 13:36:02 GMT', 'Access-Control-Allow-Methods': 'GET, HEAD, OPTIONS', 'Server': 'nginx', 'x-mbx-uuid': 'f4e20410-8623-4bbd-875e-ef61eb65706a', 'x-mbx-used-weight': '8', 'x-mbx-used-weight-1m': '8', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains', 'X-Frame-Options': 'SAMEORIGIN', 'X-Xss-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'Content-Security-Policy': "default-src 'self'", 'X-Content-Security-Policy': "default-src 'self'", 'X-WebKit-CSP': "default-src 'self'", 'Cache-Control': 'no-cache, no-store, must-revalidate', 'Pragma': 'no-cache', 'Expires': '0', 'Access-Control-Allow-Origin': '*', 'X-Cache': 'Miss from cloudfront', 'Via': '1.1 cf66796b663642e35e0aca515f0d4a6e.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'MAD56-P2', 'X-Amz-Cf-Id': 'p7nBbWt11THxJEa2QRi

In [8]:
print("Terminado")

Terminado
